first import necessary modules

In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

In [2]:
import sys
import os

# Get the absolute path to your src folder
module_path = os.path.abspath(os.path.join('..', 'src'))

if module_path not in sys.path:
    sys.path.append(module_path)

from cleaning_and_helpers import clean_rbcl_data


In [3]:
pd.set_option('display.max_columns', None)

Import and clean data frames from all projects

In [4]:
# sky islands data
# TODO update filepath when sky islands rbcl data is ready
df_si = clean_rbcl_data('../data/taxonomic/SIpollen_tax.csv',
                          '../../pollenGeolocation_saved/SIbeeRBCL_tax.csv')
df_si["Project"] = "SI"
# ffar data
# TODO update filepath when ffar rbcl data is ready
df_ffar = clean_rbcl_data('../data/taxonomic/FFARpollen_tax.csv',
                          '../../pollenGeolocation_saved/FFARbeeRBCL_tax.csv')
df_ffar["Project"] = "FFAR"
# ncasi data
# TODO update filepath when ffar rbcl data is ready
df_ncasi = clean_rbcl_data('../data/taxonomic/NCASIpollen_tax.csv',
                          '../../pollenGeolocation_saved/NCASIbeeRBCL_tax.csv',
                          format="ISO-8859-1")
df_ncasi["Project"] = "NCASI"

In [5]:
# Extract rbcl columns
SI_cols = [col for col in df_si if col.startswith('RBCL')]
SI_df = df_si[SI_cols]

FFAR_cols = [col for col in df_ffar if col.startswith('RBCL')]
FFAR_df = df_ffar[FFAR_cols]

NCASI_cols = [col for col in df_ncasi if col.startswith('RBCL')]
NCASI_df = df_ncasi[NCASI_cols]

# Get union of all features
all_features = sorted(set(SI_df.columns).union(NCASI_df.columns).union(FFAR_df.columns))

# Reindex all with full set, filling missing with 0
SI_df = SI_df.reindex(columns=all_features, fill_value=0)
NCASI_df = NCASI_df.reindex(columns=all_features, fill_value=0)
FFAR_df = FFAR_df.reindex(columns=all_features, fill_value=0)

# Convert to numpy
SI_X = SI_df.to_numpy()
np.save('../../pollenGeolocation_saved/SI_X_tax.npy', SI_X)
NCASI_X = NCASI_df.to_numpy()
np.save('../../pollenGeolocation_saved/NCASI_X_tax.npy', NCASI_X)
FFAR_X = FFAR_df.to_numpy()
np.save('../../pollenGeolocation_saved/FFAR_X_tax.npy', FFAR_X)


In [6]:
print("SI_X:", SI_X.shape)
print("NCASI_X:", NCASI_X.shape)
print("FFAR_X:", FFAR_X.shape)


SI_X: (228, 185)
NCASI_X: (176, 185)
FFAR_X: (1178, 185)


In [9]:
# Now need to extract lat_long values for y

In [7]:
## save out the x column names
np.save('../../pollenGeolocation_saved/X_columns_tax.npy', SI_df.columns)

In [10]:
# saving out targets for modeling
lat = np.array(df_si['Lat'])
long = np.array(df_si['Long'])
y = np.column_stack((lat, long))
np.save('../../pollenGeolocation_saved/SI_y.npy', y)

now we will set up a df where we can join these datasets using concat

In [11]:
# df_all = pd.concat([df_si,df_ffar,df_ncasi], axis=0, ignore_index=True)

# # this introduces NAs again where the RBCL sequences of the SI dataset were not detected in the FFAR
# # data and vice versa; use .fillna(0) to change those to 0s
# df_all = df_all.fillna(0)
# df_all.to_csv('../pollenGeolocation_saved/ALLbeeRBCL.csv')

In [13]:
## need to keep data separate for train/test split and will join afterwards
# saving out lat and long coords
df_si['lat_long'] = [', '.join(str(x) for x in y) for y in map(tuple, df_si[['Lat', 'Long']].values)]
np.save('../../pollenGeolocation_saved/si_latlong.npy', df_si['lat_long'])

df_ffar['lat_long'] = [', '.join(str(x) for x in y) for y in map(tuple, df_ffar[['Lat', 'Long']].values)]
np.save('../../pollenGeolocation_saved/ffar_latlong.npy', df_ffar['lat_long'])

df_ncasi['lat_long'] = [', '.join(str(x) for x in y) for y in map(tuple, df_ncasi[['Lat', 'Long']].values)]
np.save('../../pollenGeolocation_saved/ncasi_latlong.npy', df_ncasi['lat_long'])

In [24]:
# # saving out lat and long coords
# df_all['lat_long'] = [', '.join(str(x) for x in y) for y in map(tuple, df_all[['Lat', 'Long']].values)]
# np.save('../pollenGeolocation_saved/latlong.npy', df_all['lat_long'])

In [14]:
## SI
rbcl_cols = [col for col in df_si if col.startswith('RBCL')]
my_cols = ['Lat', 'Long', 'lat_long'] + rbcl_cols
df_si = df_si[my_cols]
df_si = df_si.reindex(columns=my_cols)

# # saving out features for modeling
# X = df_si.drop('Lat',axis=1)
# X = X.drop('Long',axis=1)
# X = X.drop('lat_long',axis=1)
# X.columns
# np.save('../pollenGeolocation_saved/SI_X_cols.npy', X.columns)
# np.save('../pollenGeolocation_saved/SI_X.npy', X)

# saving out targets for modeling
lat = np.array(df_si['Lat'])
long = np.array(df_si['Long'])
y = np.column_stack((lat, long))
np.save('../../pollenGeolocation_saved/SI_y.npy', y)

In [16]:
## FFAR
rbcl_cols = [col for col in df_ffar if col.startswith('RBCL')]
my_cols = ['Lat', 'Long', 'lat_long'] + rbcl_cols
df_ffar = df_ffar[my_cols]
df_ffar = df_ffar.reindex(columns=my_cols)

# # saving out features for modeling
# X = df_ffar.drop('Lat',axis=1)
# X = X.drop('Long',axis=1)
# X = X.drop('lat_long',axis=1)
# X.columns
# np.save('../pollenGeolocation_saved/FFAR_X_cols.npy', X.columns)
# np.save('../pollenGeolocation_saved/FFAR_X.npy', X)

# saving out targets for modeling
lat = np.array(df_ffar['Lat'])
long = np.array(df_ffar['Long'])
y = np.column_stack((lat, long))
np.save('../../pollenGeolocation_saved/FFAR_y.npy', y)

In [17]:
## NCASI
rbcl_cols = [col for col in df_ncasi if col.startswith('RBCL')]
my_cols = ['Lat', 'Long', 'lat_long'] + rbcl_cols
df_ncasi = df_ncasi[my_cols]
df_ncasi = df_ncasi.reindex(columns=my_cols)

# # saving out features for modeling
# X = df_ncasi.drop('Lat',axis=1)
# X = X.drop('Long',axis=1)
# X = X.drop('lat_long',axis=1)
# X.columns
# np.save('../pollenGeolocation_saved/NCASI_X_cols.npy', X.columns)
# np.save('../pollenGeolocation_saved/NCASI_X.npy', X)

# saving out targets for modeling
lat = np.array(df_ncasi['Lat'])
long = np.array(df_ncasi['Long'])
y = np.column_stack((lat, long))
np.save('../../pollenGeolocation_saved/NCASI_y.npy', y)

In [28]:
# # saving out features for modeling
# X = df_all.drop('Lat',axis=1)
# X = X.drop('Long',axis=1)
# X = X.drop('lat_long',axis=1)
# X.columns
# np.save('../pollenGeolocation_saved/X_cols.npy', X.columns)
# np.save('../pollenGeolocation_saved/X.npy', X)

In [29]:
# # saving out targets for modeling
# lat = np.array(df_all['Lat'])
# long = np.array(df_all['Long'])
# y = np.column_stack((lat, long))
# np.save('../pollenGeolocation_saved/y.npy', y)